In [1]:
# ============================================================
# BASELINE MODELS — TF-IDF+SVM & mBERT
# ============================================================
import os, gc, json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from torch.utils.data import Dataset
from collections import Counter

from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import f1_score, accuracy_score, classification_report

from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, DataCollatorWithPadding,
)

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

os.makedirs("../results/models", exist_ok=True)
os.makedirs("../results/analysis", exist_ok=True)

Device: cuda
GPU: NVIDIA GeForce RTX 2050


In [2]:
# ============================================================
# Load Test Data (B1: URL Holdout)
# ============================================================
train_df = pd.read_csv("../data/processed/train_hard2.csv")
test_df  = pd.read_csv("../data/processed/test_hard2.csv")

labels = ['normal', 'promo', 'smish']
label2id = {l: i for i, l in enumerate(labels)}
id2label = {i: l for l, i in label2id.items()}

print(f"Train: {train_df.shape}")
print(f"Test : {test_df.shape}")
print(f"Labels: {label2id}")
print(f"\nTest dist:\n{test_df['label'].value_counts()}")

Train: (4898, 6)
Test : (2107, 6)
Labels: {'normal': 0, 'promo': 1, 'smish': 2}

Test dist:
label
smish     1267
normal     498
promo      342
Name: count, dtype: int64


In [3]:
# ============================================================
# BASELINE 1: TF-IDF + Linear SVM
# ============================================================
print("=" * 60)
print("BASELINE 1: TF-IDF + Linear SVM")
print("=" * 60)

# Pipeline
svm_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.9,
        sublinear_tf=True,
    )),
    ('clf', LinearSVC(
        C=1.0,
        class_weight='balanced',
        random_state=SEED,
        max_iter=2000,
    )),
])

# Train
print("Training...")
svm_pipeline.fit(train_df['text_clean'], train_df['label'])
print("Training done ✅")

# Predict
svm_pred = svm_pipeline.predict(test_df['text_clean'])

# Metrics
svm_f1 = f1_score(test_df['label'], svm_pred, average='macro')
svm_acc = accuracy_score(test_df['label'], svm_pred)

# Smish recall
smish_mask = (test_df['label'] == 'smish').values
svm_smish_recall = (svm_pred[smish_mask] == 'smish').mean()

print(f"\nMacro F1     : {svm_f1:.4f}")
print(f"Accuracy     : {svm_acc:.4f}")
print(f"Smish Recall : {svm_smish_recall:.4f}")
print("\n" + "=" * 60)
print("Classification Report:")
print("=" * 60)
print(classification_report(test_df['label'], svm_pred))

BASELINE 1: TF-IDF + Linear SVM
Training...
Training done ✅

Macro F1     : 0.5888
Accuracy     : 0.5463
Smish Recall : 0.2589

Classification Report:
              precision    recall  f1-score   support

      normal       0.38      0.99      0.55       498
       promo       0.69      0.97      0.81       342
       smish       0.98      0.26      0.41      1267

    accuracy                           0.55      2107
   macro avg       0.69      0.74      0.59      2107
weighted avg       0.79      0.55      0.51      2107



In [4]:
import gc
gc.collect()
torch.cuda.empty_cache()

MBERT_NAME = "bert-base-multilingual-cased"
print(f"Loading {MBERT_NAME}...")

mbert_tokenizer = AutoTokenizer.from_pretrained(MBERT_NAME)
mbert_model = AutoModelForSequenceClassification.from_pretrained(
    MBERT_NAME,
    num_labels=3,
    id2label=id2label,
    label2id=label2id,
)

total_p = sum(p.numel() for p in mbert_model.parameters())
train_p = sum(p.numel() for p in mbert_model.parameters() if p.requires_grad)
print(f"Total params    : {total_p:,}")
print(f"Trainable params: {train_p:,} (100% — full fine-tuning)")

Loading bert-base-multilingual-cased...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Total params    : 177,855,747
Trainable params: 177,855,747 (100% — full fine-tuning)


In [5]:
class SMSDataset(Dataset):
    def __init__(self, df, tokenizer, label2id, max_len=128):
        self.texts = df['text_clean'].astype(str).tolist()
        self.labels = [label2id[l] for l in df['label']]
        self.tokenizer = tokenizer
        self.max_len = max_len
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors=None,
        )
        enc['labels'] = self.labels[idx]
        return enc


train_ds = SMSDataset(train_df, mbert_tokenizer, label2id, 128)
test_ds  = SMSDataset(test_df,  mbert_tokenizer, label2id, 128)

print(f"Train: {len(train_ds)}")
print(f"Test : {len(test_ds)}")

Train: 4898
Test : 2107


In [6]:
from sklearn.metrics import f1_score, accuracy_score

# Class weights
lc = Counter(train_df['label'].map(label2id))
tot = sum(lc.values())
cw = {i: tot / (3 * lc[i]) for i in range(3)}
print(f"Label counts : {dict(lc)}")
print(f"Class weights: {cw}")

class WeightedTrainer(Trainer):
    def __init__(self, class_weights=None, **kwargs):
        super().__init__(**kwargs)
        self.class_weights = class_weights
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        y = inputs.pop('labels')
        outputs = model(**inputs)
        logits = outputs.logits
        w = torch.tensor(
            [self.class_weights[i] for i in range(len(self.class_weights))],
            device=logits.device, dtype=torch.float32,
        )
        loss = nn.CrossEntropyLoss(weight=w)(logits, y)
        return (loss, outputs) if return_outputs else loss


def compute_metrics(eval_pred):
    logits, y_true = eval_pred
    y_pred = np.argmax(logits, axis=-1)
    return {
        "macro_f1": f1_score(y_true, y_pred, average='macro'),
        "accuracy": accuracy_score(y_true, y_pred),
    }

print("✅ Classes ready")

Label counts : {2: 1542, 0: 1990, 1: 1366}
Class weights: {0: 0.8204355108877722, 1: 1.1952171791117618, 2: 1.0587980977086036}
✅ Classes ready


In [7]:
training_args = TrainingArguments(
    output_dir="../results/models/mbert_full",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    weight_decay=0.05,
    warmup_steps=0.2,
    max_grad_norm=0.5,
    label_smoothing_factor=0.1,
    lr_scheduler_type="cosine",
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_macro_f1",
    greater_is_better=True,
    logging_steps=100,
    save_total_limit=1,
    report_to="none",
    fp16=True,
    seed=SEED,
)

print("✅ Training args ready")

✅ Training args ready


In [8]:
mbert_trainer = WeightedTrainer(
    model=mbert_model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    processing_class=mbert_tokenizer,
    data_collator=DataCollatorWithPadding(mbert_tokenizer),
    compute_metrics=compute_metrics,
    class_weights=cw,
)

print("✅ Trainer ready")
print(f"Train size: {len(mbert_trainer.train_dataset)}")
print(f"Eval size : {len(mbert_trainer.eval_dataset)}")

✅ Trainer ready
Train size: 4898
Eval size : 2107


In [9]:
# ============================================================
# TRAIN mBERT (Full Fine-tuning)
# ============================================================
print("=" * 60)
print("STARTING mBERT TRAINING")
print("=" * 60)
print("⏱️ Expected time: 30-40 minutes")
print("⚠️ Do not interrupt!")
print("=" * 60)

mbert_trainer.train()

print("\n✅ mBERT Training done!")

STARTING mBERT TRAINING
⏱️ Expected time: 30-40 minutes
⚠️ Do not interrupt!


Epoch,Training Loss,Validation Loss,Macro F1,Accuracy
1,0.614224,4.983630,0.381948,0.397247
2,0.200178,2.815310,0.644758,0.638348
3,0.032602,3.930244,0.548712,0.542003


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte


✅ mBERT Training done!


In [10]:
# ============================================================
# STEP 1: List available checkpoints
# ============================================================
import os

ckpt_dir = "../results/models/mbert_full"
print("=" * 60)
print("AVAILABLE CHECKPOINTS")
print("=" * 60)

if os.path.exists(ckpt_dir):
    for d in sorted(os.listdir(ckpt_dir)):
        full_path = os.path.join(ckpt_dir, d)
        if d.startswith("checkpoint"):
            print(f"  📁 {d}")
        elif os.path.isfile(full_path):
            print(f"  📄 {d}")
else:
    print(f"⚠️ {ckpt_dir} not found!")
    print("Check: ../results/models/")
    print(os.listdir("../results/models/"))

AVAILABLE CHECKPOINTS
  📁 checkpoint-614


In [11]:
# ============================================================
# STEP 2: Load BEST checkpoint (Epoch 2) & evaluate
# ============================================================
import gc
import numpy as np
import torch
from transformers import AutoModelForSequenceClassification, DataCollatorWithPadding

gc.collect()
torch.cuda.empty_cache()

BEST_CKPT = "../results/models/mbert_full/checkpoint-614"
print(f"Loading best checkpoint: {BEST_CKPT}")
print("=" * 60)

best_mbert = AutoModelForSequenceClassification.from_pretrained(
    BEST_CKPT,
    num_labels=3,
    id2label=id2label,
    label2id=label2id,
)
best_mbert.to(device)

# New trainer
mbert_best_trainer = WeightedTrainer(
    model=best_mbert,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    processing_class=mbert_tokenizer,
    data_collator=DataCollatorWithPadding(mbert_tokenizer),
    compute_metrics=compute_metrics,
    class_weights=cw,
)

# Evaluate
best_res = mbert_best_trainer.evaluate(test_ds, metric_key_prefix="test")
mbert_f1 = best_res['test_macro_f1']

print(f"\n{'='*60}")
print(f"mBERT BEST CHECKPOINT (Epoch 2)")
print(f"{'='*60}")
print(f"Test Macro F1: {mbert_f1:.4f}")
print(f"Test Accuracy: {best_res['test_accuracy']:.4f}")

# Smish recall
preds_output = mbert_best_trainer.predict(test_ds)
y_pred_mbert = np.argmax(preds_output.predictions, axis=-1)
y_true = preds_output.label_ids

smish_idx = label2id['smish']
smish_mask = y_true == smish_idx
mbert_smish_recall = (y_pred_mbert[smish_mask] == smish_idx).mean()

print(f"Smish Recall : {mbert_smish_recall:.4f}")
print("\nClassification Report:")
print(classification_report(y_true, y_pred_mbert, target_names=labels))

Loading best checkpoint: ../results/models/mbert_full/checkpoint-614


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch,Macro F1,Accuracy
No log,2.815310,0,0.644758,0.638348



mBERT BEST CHECKPOINT (Epoch 2)
Test Macro F1: 0.6448
Test Accuracy: 0.6383


Smish Recall : 0.4088

Classification Report:
              precision    recall  f1-score   support

      normal       0.65      1.00      0.79       498
       promo       0.41      0.96      0.57       342
       smish       0.98      0.41      0.58      1267

    accuracy                           0.64      2107
   macro avg       0.68      0.79      0.64      2107
weighted avg       0.81      0.64      0.63      2107



In [12]:
# ============================================================
# STEP 3A: mBERT + LoRA Setup
# ============================================================
import gc
import torch
from peft import LoraConfig, get_peft_model, TaskType

gc.collect()
torch.cuda.empty_cache()

# Fresh mBERT
mbert_lora_base = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-multilingual-cased",
    num_labels=3,
    id2label=id2label,
    label2id=label2id,
)

# LoRA config
lora_cfg_mbert = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=16,
    lora_alpha=32,
    lora_dropout=0.2,
    bias="none",
    target_modules=["query", "value"],
)

mbert_lora_model = get_peft_model(mbert_lora_base, lora_cfg_mbert)
mbert_lora_model.print_trainable_parameters()
print("✅ mBERT + LoRA ready")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 592,131 || all params: 178,447,878 || trainable%: 0.3318
✅ mBERT + LoRA ready


In [13]:
# ============================================================
# STEP 3B: Training Arguments for mBERT + LoRA
# ============================================================
training_args_mbert_lora = TrainingArguments(
    output_dir="../results/models/mbert_lora_r16",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=1,
    num_train_epochs=3,
    weight_decay=0.05,
    warmup_steps=0.2,
    max_grad_norm=0.5,
    label_smoothing_factor=0.1,
    lr_scheduler_type="cosine",
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_macro_f1",
    greater_is_better=True,
    logging_steps=100,
    save_total_limit=1,
    report_to="none",
    fp16=True,
    optim="adamw_torch_fused",
    seed=SEED,
)
print("✅ Training args ready")

✅ Training args ready


In [14]:
# ============================================================
# STEP 3C: Create Trainer
# ============================================================
mbert_lora_trainer = WeightedTrainer(
    model=mbert_lora_model,
    args=training_args_mbert_lora,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    processing_class=mbert_tokenizer,
    data_collator=DataCollatorWithPadding(mbert_tokenizer),
    compute_metrics=compute_metrics,
    class_weights=cw,
)
print("✅ mBERT + LoRA Trainer ready")
print(f"Train size: {len(mbert_lora_trainer.train_dataset)}")
print(f"Eval size : {len(mbert_lora_trainer.eval_dataset)}")

✅ mBERT + LoRA Trainer ready
Train size: 4898
Eval size : 2107


In [15]:
# ============================================================
# STEP 3D: TRAIN mBERT + LoRA
# ============================================================
print("=" * 60)
print("STARTING mBERT + LoRA TRAINING")
print("⏱️ Expected: 10 minutes")
print("=" * 60)

mbert_lora_trainer.train()

print("\n✅ mBERT + LoRA Training done!")

STARTING mBERT + LoRA TRAINING
⏱️ Expected: 10 minutes


Epoch,Training Loss,Validation Loss,Macro F1,Accuracy
1,0.766363,0.927196,0.516185,0.535833
2,0.277351,1.819266,0.394092,0.398671
3,0.231244,1.960306,0.390503,0.395349



✅ mBERT + LoRA Training done!


In [16]:
# ============================================================
# FINAL BASELINE COMPARISON — 5 Models
# ============================================================
import json

print("=" * 100)
print(" " * 25 + "FINAL BASELINE COMPARISON")
print("=" * 100)
print(f"{'Model':32s} | {'Type':14s} | {'Trainable':>15s} | {'Test F1':>10s} | {'Smish Recall':>13s}")
print("-" * 100)

final_baselines = [
    ("TF-IDF + LR",          "Classical",     "—",              0.5963, 0.28),
    ("TF-IDF + SVM",         "Classical",     "—",              float(svm_f1), float(svm_smish_recall)),
    ("mBERT (Full FT)",      "Transformer",   "178M (100%)",    float(mbert_f1), float(mbert_smish_recall)),
    ("MuRIL (Full FT)",      "Indic",         "237M (100%)",    0.4690, 0.0647),
    ("XLM-R + LoRA (Ours)",  "Proposed",      "3.2M (1.15%)",   0.7437, 0.5912),
]

for name, type_, params, f1, recall in final_baselines:
    marker = " 🏆" if "Ours" in name else ""
    print(f"{name:32s} | {type_:14s} | {params:>15s} | {f1:>10.4f} | {recall:>13.4f}{marker}")

print("=" * 100)

all_baselines_final = {
    "TF-IDF + LR": 0.5963,
    "TF-IDF + SVM": float(svm_f1),
    "mBERT (Full FT)": float(mbert_f1),
    "MuRIL (Full FT)": 0.4690,
    "XLM-R + LoRA (Ours)": 0.7437,
}
with open("../results/all_baselines_final.json", "w") as f:
    json.dump(all_baselines_final, f, indent=2)

print("\n✅ Saved: all_baselines_final.json")

                         FINAL BASELINE COMPARISON
Model                            | Type           |       Trainable |    Test F1 |  Smish Recall
----------------------------------------------------------------------------------------------------
TF-IDF + LR                      | Classical      |               — |     0.5963 |        0.2800
TF-IDF + SVM                     | Classical      |               — |     0.5888 |        0.2589
mBERT (Full FT)                  | Transformer    |     178M (100%) |     0.6448 |        0.4088
MuRIL (Full FT)                  | Indic          |     237M (100%) |     0.4690 |        0.0647
XLM-R + LoRA (Ours)              | Proposed       |    3.2M (1.15%) |     0.7437 |        0.5912 🏆

✅ Saved: all_baselines_final.json
